In [ ]:
from eelbrain import *
import glob
import mne
import numpy as np

In [ ]:
subjs = glob.glob('./analysis_800ms_epochs/looming*-epo.fif')

In [ ]:
conds_to_compare = ['1002', '1003']

In [ ]:
tstep = 1. / 1000
n_times = 801
time = UTS(-0.100, tstep, n_times)

sensor = Sensor.from_montage('easycap-M1')[:64]

rows = []

for subj in subjs:
    subj_epochs = mne.read_epochs(subj, preload=True, verbose=False)
    subj_epochs = subj_epochs.drop_channels('STI')

    for i in range(len(subj_epochs)):
        if (subj_epochs[i].events[:, 2][0] == int(conds_to_compare[0])) or (subj_epochs[i].events[:, 2][0] == int(conds_to_compare[1])):

            subject = int(subj.split('-')[0][-3:])

            condition = str(subj_epochs[i].events[:, 2][0])

            eeg_deviant = load.fiff.epochs_ndvar(subj_epochs[i])
            eeg_std = load.fiff.epochs_ndvar(subj_epochs[i-1])
            eeg = eeg_deviant - eeg_std
            eeg.name = 'EEG'

            rows.append([subject, condition, eeg])

ds = Dataset.from_caselist(['subject','condition', 'eeg'], rows)
ds['subject'].random = True
print(ds.summary())

In [ ]:
p = plot.SensorMap(ds['eeg'], connectivity=True)

In [ ]:
res = testnd.TTestRelated(
    'eeg', 'condition', conds_to_compare[0], conds_to_compare[1], match='subject', ds=ds,
    pmin=0.05,  # Use uncorrected p = 0.05 as threshold for forming clusters
    tstart=0,  # Find clusters in the time window from 100 ...
    tstop=0.700,  # ... to 600 ms
    # mintime=0.1,
    # minsource=2
)

In [ ]:
p = plot.TopoButterfly(res, clip='circle')
p.plot_colorbar()
p.set_time(0.470)

In [ ]:
clusters = res.find_clusters(0.05)
print(clusters)

In [ ]:
a = clusters['id'][:]

cluster_ndvars = []

for i in range(len(a)):
    cluster = res.cluster(a[i])
    cluster_ndvars.append(cluster)
    p = plot.TopoArray(cluster, interpolation='nearest')
    p.set_topo_ts(0.2, 0.3, 0.4)

In [ ]:
masks = []

for i in range(len(cluster_ndvars)):

    # sensor x sample plot with topogrpahy
    mask = cluster_ndvars[i] != 0
    masks.append(mask)
    p = plot.TopoArray(mask, cmap='Wistia')
    p.set_topo_ts(.280, 0.380, 0.470)

    # topography map for spatial extent of each cluster
    roi = mask.any('time')
    p = plot.Topomap(roi, cmap='Wistia')
    
    ds['cluster_timecourse'] = ds['eeg'].mean(roi)
    p = plot.UTSStat('cluster_timecourse', 'condition', match='subject', ds=ds, frame='t', title='Compare conditions (all channels within cluster)')
    # mark the duration of the spatio-temporal cluster
    p.set_clusters(clusters[[i]])

    # mark significant sensors in topographic map of difference between conditions
    time_window = (clusters[0, 'tstart'], clusters[0, 'tstop'])
    c1_topo = res.c1_mean.mean(time=time_window)
    c0_topo = res.c0_mean.mean(time=time_window)
    diff_topo = res.difference.mean(time=time_window)
    p = plot.Topomap([c1_topo, c0_topo, diff_topo], axtitle=[conds_to_compare[0], conds_to_compare[1], f'{conds_to_compare[0]}-{conds_to_compare[1]}'], ncol=3)
    p.mark_sensors(roi, -1)

#### Temporal cluster-based test

In [ ]:
ds['eeg_fz'] = ds['eeg'].sub(sensor='Fz')
print(ds.summary())

res_timecoure = testnd.TTestRelated(
    'eeg_fz', 'condition', conds_to_compare[0], conds_to_compare[1], match='subject', ds=ds,
    pmin=0.05,  # Use uncorrected p = 0.05 as threshold for forming clusters
    tstart=0,  # Find clusters in the time window from 100 ...
    tstop=0.700,  # ... to 600 ms
    # mintime=0.1
)
clusters_fz = res_timecoure.find_clusters(0.05)
print(clusters_fz)

p = plot.UTSStat('eeg_fz', 'condition', match='subject', ds=ds, frame='t', title='Compare conditions (Fz)')
p.set_clusters(clusters_fz, y=0.25e-6)